# Evaporating Universe — Paper I
## NB05_MCMC_A1: Chain Analysis + Cross-Check — Run A1

**Purpose:** Reproducibility & Transparency for Referees

| Item | Value |
|:-----|:------|
| Run | A1 — EU params SAMPLED, background-only IDE |
| EU params | SAMPLED — ε_IR, z_trans, b are free parameters with priors |
| IDE perturbations | `eu_has_ide_perturbations` = value from YAML (verified in §3) |
| Datasets | Planck NPIPE CamSpec TTTEEE + lowl TT/EE + lensing + DESI DR2 + Pantheon+ |
| Chains | 8 MPI chains, R-1 target < 0.01 |
| Infrastructure | AWS m7a.48xlarge (192 vCPU) |
| Theory code | CLASS v3.3.4 + EU patch vf1.3 |

**A1 vs A2:** A1 samples EU parameters with background-only IDE (`eu_has_ide_perturbations=0`). A2 enables full IDE perturbation source terms (`eu_has_ide_perturbations=1`). Comparing A1 and A2 tests whether IDE perturbation source terms affect the posterior when EU params are free.

**Workflow:**
1. §1: Upload ALL files (patch + chains + YAML) — one dialog
2. §2: Environment setup (Cobaya + CLASS EU + likelihoods) — ~15 min
3. §3: Display YAML configuration (exact run conditions + perturbation flag)
4. §4: Convergence diagnostics (R-1 + ESS + trace plots)
5. §5: Corner plots (GetDist)
6. §6: Parameter table + best-fit
7. §7: Cross-check — evaluate χ² at best-fit point
8. §8: Export JSON (referee-grade)
9. §9: Download

**Referee instructions:** Upload files in §1, then Run All. §2 takes ~15 min. §7 takes ~2 min.

**No hardcoded theory values. No fallbacks. All parameters extracted from YAML.**

---


In [ ]:
# ============================================================
# §1. UPLOAD ALL FILES
# ============================================================
# Upload everything needed in ONE step:
#   - patch_class.py (EU modification for CLASS)
#   - 8 chain files (eu_NB05A.1.txt ... eu_NB05A.8.txt)
#   - eu_NB05A.updated.yaml (run configuration)
#
# Tip: zip all 10 files into one archive for convenience.
# ============================================================
import os, zipfile

CHAIN_ROOT = 'eu_NB05A'
N_CHAINS = 8

ALL_REQUIRED = (
    ['patch_class.py']
    + [f'{CHAIN_ROOT}.{i}.txt' for i in range(1, N_CHAINS + 1)]
    + [f'{CHAIN_ROOT}.updated.yaml']
    + [f'{CHAIN_ROOT}.checkpoint']
)

print('=' * 60)
print('REQUIRED FILES (upload all at once)')
print('  Include .checkpoint file for convergence verification')
print('=' * 60)
for f in ALL_REQUIRED:
    status = '\u2705' if os.path.exists(f'/content/{f}') else '\u274c'
    print(f'  {status} {f}')

missing = [f for f in ALL_REQUIRED if not os.path.exists(f'/content/{f}')]
if missing:
    print(f'\n  {len(missing)} files missing — opening upload dialog...')
    print(f'  Tip: zip all files and upload the zip.\n')
    from google.colab import files as _files
    uploaded = _files.upload()
    for name, content in uploaded.items():
        with open(f'/content/{name}', 'wb') as f:
            f.write(content)
        if name.endswith('.zip'):
            with zipfile.ZipFile(f'/content/{name}', 'r') as zf:
                zf.extractall('/content')
            print(f'  [OK] {name} extracted')
        else:
            print(f'  [OK] {name}')

# Final verification
print(f'\n  Verification:')
all_ok = True
for f in ALL_REQUIRED:
    exists = os.path.exists(f'/content/{f}')
    status = '\u2705' if exists else '\u274c MISSING'
    print(f'    {status} {f}')
    if not exists:
        all_ok = False

assert all_ok, 'FATAL: Not all files present. Re-run this cell.'
print(f'\n[OK] All {len(ALL_REQUIRED)} files present')


---


In [ ]:
# ============================================================
# §2. ENVIRONMENT SETUP
# ============================================================
# Installs: Cobaya, GetDist, CLASS EU (patched), Planck+DESI+SNe
# Time: ~15 min on Colab (compile + download)
# ============================================================
import os, subprocess, sys, time
import numpy as np

t0_setup = time.time()

PACKAGES_PATH = '/content/packages'
CLASS_DIR = '/content/class_eu'
PATCH_SCRIPT = '/content/patch_class.py'

# -- Step 1: Python packages --
print('=' * 60)
print('[1/5] Installing Python packages...')
os.system('pip install -q cobaya getdist pyyaml matplotlib numpy scipy cython')

import cobaya
print(f'  [OK] Cobaya {cobaya.__version__}')
import getdist
print(f'  [OK] GetDist {getdist.__version__}')

# -- Step 2: Clone CLASS v3.3.4 --
print('\n[2/5] Cloning CLASS v3.3.4...')
if not os.path.isdir(CLASS_DIR):
    os.system(f'git clone --branch v3.3.4 --depth 1 https://github.com/lesgourg/class_public.git {CLASS_DIR} 2>&1 | tail -2')
    print('  [OK] CLASS cloned')
else:
    print('  [SKIP] CLASS already present')

# -- Step 3: Apply EU patch --
print('\n[3/5] Applying EU patches to CLASS...')
assert os.path.exists(PATCH_SCRIPT), f'FATAL: {PATCH_SCRIPT} not found. Re-run §1.'
os.system(f'cd /content && python3 {PATCH_SCRIPT} {CLASS_DIR}')

# -- Step 4: Compile CLASS + install classy --
print('\n[4/5] Compiling CLASS-EU...')
os.system(f'cd {CLASS_DIR} && make clean > /dev/null 2>&1; make -j$(nproc) 2>&1 | tail -3')
assert os.path.exists(f'{CLASS_DIR}/libclass.a'), 'FATAL: libclass.a not built!'
os.system(f'cd {CLASS_DIR} && pip install -q --no-build-isolation . 2>&1 | tail -2')

# Smoke test
result = subprocess.run(
    [sys.executable, '-c', """
from classy import Class
c = Class()
c.set({"output":"tCl", "l_max_scalars":100,
       "100*theta_s": 1.0411,
       "omega_b": 0.02237, "omega_cdm": 0.12,
       "eu_epsilon_ir": 0.04264, "eu_z_trans": 5.986, "eu_b": 0.5278,
       "Omega_Lambda": 0, "w0_fld": -1.0, "wa_fld": 0, "cs2_fld": 1.0,
       "eu_has_ide_perturbations": 0,
       "N_ur": 2.0328, "N_ncdm": 1, "m_ncdm": 0.0589})
c.compute()
print(f"H0={c.h()*100:.2f}")
c.struct_cleanup(); c.empty()
"""],
    capture_output=True, text=True
)
assert 'H0=' in result.stdout, f'FATAL: CLASS smoke test failed!\n{result.stderr}'
print(f'  [OK] CLASS-EU smoke test: {result.stdout.strip()}')

# -- Step 5: Install likelihood data --
print('\n[5/5] Installing likelihood data (Planck NPIPE + DESI DR2 + Pantheon+)...')
os.system(f"""cobaya-install \\
    planck_NPIPE_highl_CamSpec.TTTEEE \\
    planck_2018_lowl.TT \\
    planck_2018_lowl.EE \\
    planck_2018_lensing.clik \\
    sn.pantheonplus \\
    bao.desi_dr2 \\
    -p {PACKAGES_PATH} 2>&1 | tail -5""")

# Install clipy for Planck lensing
os.system(f'cobaya-install planck_2018_lensing.clik -p {PACKAGES_PATH} 2>&1 | tail -3')

dt_setup = time.time() - t0_setup
print(f'\n{"="*60}')
print(f'[OK] Setup complete in {dt_setup:.0f}s')
print(f'  CLASS-EU: {CLASS_DIR}')
print(f'  Packages: {PACKAGES_PATH}')
print(f'  Cobaya: {cobaya.__version__}')
print(f'={"="*59}')


---


In [ ]:
# ============================================================
# §3. CONFIGURATION VERIFICATION
# ============================================================
# Display the YAML that Cobaya actually used.
# This is the receipt — proof of exact run conditions.
# ============================================================
import yaml, glob

print('=' * 60)
print('RUN CONFIGURATION (from updated.yaml)')
print('=' * 60)

yaml_path = f'/content/{CHAIN_ROOT}.updated.yaml'
with open(yaml_path) as f:
    config = yaml.safe_load(f)

theory = config.get('theory', {}).get('classy', {})
extra_args = theory.get('extra_args', {})
likelihoods = list(config.get('likelihood', {}).keys())
sampler_cfg = config.get('sampler', {}).get('mcmc', {})

# EU perturbation flag — extracted from YAML, NOT hardcoded
eu_pert_flag = extra_args.get('eu_has_ide_perturbations', 'NOT_SET')

# EU params may be in params (fixed) or extra_args
params = config.get('params', {})
eu_param_vals = {}
for ep in ['eu_epsilon_ir', 'eu_z_trans', 'eu_b']:
    if ep in extra_args:
        eu_param_vals[ep] = extra_args[ep]
    elif ep in params and isinstance(params[ep], dict):
        eu_param_vals[ep] = params[ep].get('value', params[ep].get('ref', None))
    elif ep in params and not isinstance(params[ep], dict):
        eu_param_vals[ep] = params[ep]
    else:
        eu_param_vals[ep] = None
if eu_pert_flag == 0:
    eu_pert_label = 'OFF (background-only IDE, perturbation source terms \u0394k=0)'
elif eu_pert_flag == 1:
    eu_pert_label = 'ON (full IDE: background + perturbation source terms)'
else:
    eu_pert_label = f'UNKNOWN ({eu_pert_flag})'

print(f'\n  Theory: CLASS {theory.get("version", "?")}')
print(f'  EU params:')
print(f'    \u03b5_IR     = {eu_param_vals.get("eu_epsilon_ir")}')
print(f'    z_trans  = {eu_param_vals.get("eu_z_trans")}')
print(f'    b        = {eu_param_vals.get("eu_b")}')
print(f'    \u03bb        = {extra_args.get("eu_lambda", "2/3 default")}')
print(f'  IDE perturbations: eu_has_ide_perturbations = {eu_pert_flag}')
print(f'    \u2192 {eu_pert_label}')
print(f'  Likelihoods ({len(likelihoods)}):')
for lik in likelihoods:
    print(f'    \u2022 {lik}')
print(f'  R-1 target: {sampler_cfg.get("Rminus1_stop", "?")}')
print(f'  Cobaya: {config.get("version", "?")}')

# Sampled vs derived
sampled = [p for p, v in params.items()
           if isinstance(v, dict) and 'prior' in v]
derived = [p for p, v in params.items()
           if isinstance(v, dict) and v.get('derived') is True]
print(f'\n  Sampled parameters ({len(sampled)}):')
for p in sampled:
    pr = params[p].get('prior', {})
    print(f'    {p}: {pr}')
print(f'\n  Derived parameters ({len(derived)}):')
for p in derived:
    print(f'    {p}')


---


In [ ]:
# ============================================================
# §4. CONVERGENCE DIAGNOSTICS + TRACE PLOTS
# ============================================================

print('=' * 60)
print('CONVERGENCE DIAGNOSTICS')
print('=' * 60)

chain_files = sorted(glob.glob(f'/content/{CHAIN_ROOT}.*.txt'))
assert len(chain_files) == N_CHAINS, f'Expected {N_CHAINS} chains, got {len(chain_files)}. No fallback.'

# Read header
with open(chain_files[0]) as f:
    header = f.readline().strip().lstrip('#').split()

# Load chains with 30% burn-in
chains_raw = []
for cf in chain_files:
    data = np.loadtxt(cf)
    burn = int(0.3 * len(data))
    chains_raw.append(data[burn:])
    print(f'  {os.path.basename(cf)}: {len(data)} rows, burn={burn}, kept={len(data)-burn}')

# Gelman-Rubin R-1
def gelman_rubin(chains, col_idx, weight_idx=0):
    chain_means, chain_vars, chain_n = [], [], []
    for c in chains:
        w = c[:, weight_idx]
        x = c[:, col_idx]
        n = np.sum(w)
        mean = np.average(x, weights=w)
        var = np.average((x - mean)**2, weights=w)
        chain_means.append(mean)
        chain_vars.append(var)
        chain_n.append(n)
    m = len(chains)
    grand_mean = np.mean(chain_means)
    n_avg = np.mean(chain_n)
    B = n_avg / (m - 1) * sum((mu - grand_mean)**2 for mu in chain_means)
    W = np.mean(chain_vars)
    if W < 1e-30:
        return np.nan
    V = (1 - 1/n_avg) * W + B / n_avg
    return V / W - 1

print(f'\n{"Parameter":<25} {"R-1":>10} {"Status":>8} {"Mean":>14} {"Std":>12}')
print('-' * 72)

all_data = np.vstack(chains_raw)
all_w = all_data[:, 0]

skip = {'weight', 'minuslogpost', 'minuslogprior', 'minuslogprior__0', 'chi2'}
r1_results = {}

for j, name in enumerate(header):
    if name in skip or name.startswith('chi2__'):
        continue
    r1 = gelman_rubin(chains_raw, j)
    mean = np.average(all_data[:, j], weights=all_w)
    std = np.sqrt(np.average((all_data[:, j] - mean)**2, weights=all_w))
    r1_results[name] = {'R-1': r1, 'mean': mean, 'std': std}
    if np.isnan(r1):
        status = '(const)'
    elif r1 < 0.01:
        status = '\u2705'
    elif r1 < 0.05:
        status = '\u26a0\ufe0f'
    else:
        status = '\u274c'
    print(f'  {name:<23} {r1:10.4f} {status:>8} {mean:14.6f} {std:12.6f}')

sampled_r1 = {k: v for k, v in r1_results.items() if not np.isnan(v['R-1'])}
converged = sum(1 for v in sampled_r1.values() if v['R-1'] < 0.01)
r1_max = max(v['R-1'] for v in sampled_r1.values())
total_weighted = int(np.sum(all_w))

print(f'\n  Converged: {converged}/{len(sampled_r1)} (R-1 < 0.01)')
print(f'  Worst R-1: {r1_max:.4f}')
print(f'  Total weighted samples: {total_weighted:,}')

# ── Effective Sample Size (ESS) via GetDist ──
from getdist import MCSamples
print(f'\n  Effective Sample Size (ESS):')
combined_gd = MCSamples(
    samples=all_data[:, 2:], weights=all_data[:, 0], loglikes=all_data[:, 1],
    names=header[2:], labels=header[2:]
)
ess_key = ['omega_b', 'omega_cdm', 'H0', 'tau_reio', 'logA', 'n_s', 'sigma8', 'S8']
for p in ess_key:
    if p in header[2:]:
        idx = header[2:].index(p)
        try:
            ess = combined_gd.getEffectiveSamplesForParamIndex(idx)
            print(f'    {p:<16} ESS = {ess:,.0f}')
        except:
            pass

# ── Trace Plots ──
import matplotlib.pyplot as plt
os.makedirs('figures', exist_ok=True)

trace_params = ['H0', 'omega_cdm', 'sigma8', 'S8', 'tau_reio', 'n_s']
trace_available = [p for p in trace_params if p in header]

fig, axes = plt.subplots(len(trace_available), 1, figsize=(14, 3 * len(trace_available)), sharex=True)
if len(trace_available) == 1:
    axes = [axes]

colors = plt.cm.tab10(np.linspace(0, 1, N_CHAINS))

for ax, pname in zip(axes, trace_available):
    col_idx = header.index(pname)
    for ci, chain in enumerate(chains_raw):
        # Thin for plotting (every 10th sample)
        step = max(1, len(chain) // 2000)
        samples = chain[::step, col_idx]
        ax.plot(samples, alpha=0.5, linewidth=0.5, color=colors[ci], label=f'Chain {ci+1}' if pname == trace_available[0] else None)
    mean_val = r1_results[pname]['mean']
    ax.axhline(mean_val, color='red', linewidth=1.5, linestyle='--', alpha=0.8)
    ax.set_ylabel(pname, fontsize=12)
    r1_val = r1_results[pname]['R-1']
    ax.set_title(f'{pname}   (R-1 = {r1_val:.4f})', fontsize=11, loc='right')

axes[0].legend(loc='upper right', ncol=4, fontsize=8)
axes[-1].set_xlabel('Sample (post burn-in, thinned)', fontsize=12)
fig.suptitle('Trace Plots \u2014 8 Independent MPI Chains', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig_NB05_A1_trace_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Trace plots saved')


---


In [ ]:
# ============================================================
# §5. CORNER PLOTS (GetDist)
# ============================================================

from getdist import MCSamples, plots
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'figure.facecolor': 'white'
})
os.makedirs('figures', exist_ok=True)

print('=' * 60)
print('CORNER PLOTS')
print('=' * 60)

# Combined sample
all_post = np.vstack([c[int(0.3*len(c)):] for c in [np.loadtxt(cf) for cf in chain_files]])
combined = MCSamples(
    samples=all_post[:, 2:], weights=all_post[:, 0], loglikes=all_post[:, 1],
    names=header[2:], labels=header[2:],
    name_tag='A1 Combined', label='EU A1'
)

# -- Cosmological triangle --
cosmo_params = ['omega_b', 'omega_cdm', 'H0', 'tau_reio', 'logA', 'n_s', 'sigma8', 'S8']
cosmo_available = [p for p in cosmo_params if p in header[2:]]

g = plots.get_subplot_plotter(width_inch=14)
g.settings.axes_fontsize = 10
g.settings.axes_labelsize = 12
g.settings.title_limit_fontsize = 10
g.triangle_plot(combined, cosmo_available, filled=True,
                contour_colors=['#2196F3'], title_limit=1)
plt.subplots_adjust(hspace=0.08, wspace=0.08)
plt.savefig('figures/fig_NB05_A1_corner_cosmo.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB05_A1_corner_cosmo.pdf', bbox_inches='tight')
plt.show()
print('[OK] Cosmological corner plot')

# -- EU-derived --
eu_params = ['H0', 'H0_LKI', 'sigma8', 'S8', 'Omega_m', 'rdrag']
eu_available = [p for p in eu_params if p in header[2:]]

g2 = plots.get_subplot_plotter(width_inch=12)
g2.settings.axes_fontsize = 11
g2.settings.axes_labelsize = 13
g2.settings.title_limit_fontsize = 11
g2.triangle_plot(combined, eu_available, filled=True,
                 contour_colors=['#4CAF50'], title_limit=1)
plt.subplots_adjust(hspace=0.08, wspace=0.08)
plt.savefig('figures/fig_NB05_A1_corner_eu.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB05_A1_corner_eu.pdf', bbox_inches='tight')
plt.show()
print('[OK] EU-derived corner plot')

# -- 1D posteriors --
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
key_params = ['H0', 'H0_LKI', 'omega_cdm', 'omega_b',
              'sigma8', 'S8', 'Omega_m', 'n_s']
for ax, pname in zip(axes.flat, key_params):
    if pname in header[2:]:
        idx = header[2:].index(pname)
        vals = all_post[:, idx + 2]
        weights = all_post[:, 0]
        ax.hist(vals, bins=60, weights=weights, density=True,
                color='#2196F3', alpha=0.7, edgecolor='none')
        mean = np.average(vals, weights=weights)
        ax.axvline(mean, color='red', linewidth=1.5, linestyle='--')
        ax.set_title(pname, fontsize=12)
        ax.set_yticks([])
plt.suptitle('C2 Baseline \u2014 1D Posteriors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig_NB05_A1_1d_posteriors.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] 1D posteriors')


---


In [ ]:
# ============================================================
# §6. PARAMETER TABLE + BEST-FIT
# ============================================================

print('=' * 60)
print('PARAMETER TABLE')
print('=' * 60)

bf_idx = np.argmin(all_data[:, header.index('minuslogpost')])
bestfit = all_data[bf_idx]

cosmo_table = ['omega_b', 'omega_cdm', 'theta_s_100', 'tau_reio',
               'logA', 'n_s', 'A_planck']
derived_table = ['H0', 'sigma8', 'S8', 'Omega_m', 'rdrag',
                 'H0_LKI', 'fcdm_z0', 'I_GKI']

print(f'\n{"Parameter":<18} {"Mean":>12} {"Std":>12} {"Best-fit":>12} {"R-1":>8}')
print('-' * 65)
print('  --- Sampled ---')
for p in cosmo_table:
    if p in r1_results:
        r = r1_results[p]
        bf = bestfit[header.index(p)] if p in header else np.nan
        r1_str = f'{r["R-1"]:.4f}' if not np.isnan(r['R-1']) else 'const'
        print(f'  {p:<16} {r["mean"]:12.6f} {r["std"]:12.6f} {bf:12.6f} {r1_str:>8}')

print('  --- Derived ---')
for p in derived_table:
    if p in r1_results:
        r = r1_results[p]
        bf = bestfit[header.index(p)] if p in header else np.nan
        r1_str = f'{r["R-1"]:.4f}' if not np.isnan(r['R-1']) else 'const'
        print(f'  {p:<16} {r["mean"]:12.6f} {r["std"]:12.6f} {bf:12.6f} {r1_str:>8}')

# Chi2 breakdown
print(f'\n  Best-fit chi2:')
chi2_keys = [h for h in header if h.startswith('chi2__')]
for k in chi2_keys:
    val = bestfit[header.index(k)]
    print(f'    {k.replace("chi2__", ""):<45} {val:10.3f}')
total_from_col = bestfit[header.index('chi2')] if 'chi2' in header else 0
print(f'    {"TOTAL":<45} {total_from_col:10.3f}')


---


In [ ]:
# ============================================================
# §7. CROSS-CHECK — Evaluate likelihood at best-fit
# ============================================================
# A1/A2: EU params are SAMPLED but only feed eu_derived.EU_Derived
# (analytical H0_LKI, fcdm_z0, I_GKI). CLASS runs as LCDM —
# EU params are NOT CLASS inputs (confirmed by input_params in YAML).
# Cross-check: remove EU params + eu_derived, evaluate CLASS as LCDM.
# ============================================================

print('=' * 60)
print('CROSS-CHECK — Likelihood evaluation at best-fit')
print('=' * 60)

try:
    from cobaya.run import run as cobaya_run
    import yaml as _yaml

    with open(yaml_path) as f:
        info_verify = _yaml.safe_load(f.read())

    info_verify['sampler'] = {'evaluate': None}
    info_verify['output'] = '/content/verify_a1'
    info_verify['force'] = True
    info_verify['packages_path'] = PACKAGES_PATH

    # Remove eu_derived (custom likelihood, not in Cobaya packages)
    info_verify['likelihood'].pop('eu_derived.EU_Derived', None)

    # Remove EU sampled params (they only fed eu_derived, not CLASS)
    EU_ONLY_PARAMS = {'eu_epsilon_ir', 'eu_z_trans', 'eu_b'}
    for ep in EU_ONLY_PARAMS:
        info_verify['params'].pop(ep, None)

    # Remove eu_derived output params
    for dp in ['H0_LKI', 'fcdm_z0', 'I_GKI']:
        info_verify['params'].pop(dp, None)

    # Set all remaining sampled params to best-fit (fixed values)
    bf_params = {}
    for p, v in list(info_verify['params'].items()):
        if isinstance(v, dict) and 'prior' in v:
            if p in header:
                bf_val = float(bestfit[header.index(p)])
                info_verify['params'][p] = {'value': bf_val}
                bf_params[p] = bf_val

    print(f'\n  Evaluating at best-fit ({len(bf_params)} params, EU removed)...')
    for p, v in bf_params.items():
        print(f'    {p:<20} = {v:.6f}')

    print(f'\n  Running Cobaya evaluate...')
    updated_v, sampler_v = cobaya_run(info_verify)

    # Extract chi2
    products = sampler_v.products()
    sample = products.get('sample', products.get('minimum', None))

    live_chi2 = {}
    row = {}
    if sample is not None:
        try:
            if hasattr(sample, 'data'):
                row = sample.data.iloc[0].to_dict()
            elif hasattr(sample, 'iloc'):
                row = sample.iloc[0].to_dict()
            for k, v in row.items():
                if str(k).startswith('chi2__'):
                    live_chi2[str(k)] = float(v)
        except Exception as e:
            print(f'  [WARN] Could not parse products: {e}')

    # Chain chi2 (exclude eu_derived which contributed 0)
    chain_chi2_total = float(total_from_col)

    # Compute live total from per-likelihood
    _per_lik = ['planck_NPIPE_highl_CamSpec.TTTEEE', 'planck_2018_lowl.TT',
                'planck_2018_lowl.EE', 'planck_2018_lensing.clik',
                'bao.desi_dr2', 'sn.pantheonplus']
    live_chi2_total = sum(live_chi2.get(f'chi2__{k}', 0) for k in _per_lik)

    # Derived params sanity
    if row:
        live_H0 = row.get('H0', float('nan'))
        chain_H0 = float(bestfit[header.index('H0')])
        print(f'\n  ━━ DERIVED PARAMS SANITY CHECK ━━')
        print(f'  H0:     chain={chain_H0:.3f}  live={live_H0:.3f}  diff={abs(chain_H0-live_H0):.3f}')

    print(f'\n  ━━ CHI-SQUARED COMPARISON (correct metric) ━━')
    print(f'  Chain best-fit χ²  = {chain_chi2_total:.3f}')
    print(f'  Live evaluation χ² = {live_chi2_total:.3f}')
    diff = abs(chain_chi2_total - live_chi2_total)
    print(f'  Difference: {diff:.3f}')

    if diff < 1.0:
        print(f'  ✅ MATCH — Chains verified (diff < 1.0)')
    elif diff < 5.0:
        print(f'  ⚠️ CLOSE — Minor numerical difference')
    else:
        print(f'  ❌ MISMATCH — Check configuration')

    chain_logpost = float(bestfit[header.index('minuslogpost')])
    chain_logprior = float(bestfit[header.index('minuslogprior')]) if 'minuslogprior' in header else 0
    live_logpost = float(row.get('minuslogpost', 0)) if row else 0

    print(f'\n  ━━ LOGPOST DETAIL (for reference) ━━')
    print(f'  Chain:  -logpost={chain_logpost:.3f} = -loglik + prior({chain_logprior:.3f})')
    print(f'  Live:   -logpost={live_logpost:.3f}')
    print(f'  Note: prior=0 in evaluate because params are fixed.')

    print(f'\n  Per-likelihood verification:')
    print(f'    {"Likelihood":<45} {"Chain":>10} {"Live":>10} {"Diff":>8}')
    print(f'    {"-"*75}')
    for col in sorted(live_chi2.keys()):
        if col in header:
            short = col.replace('chi2__', '')
            chain_val = float(bestfit[header.index(col)])
            live_val = live_chi2[col]
            d = abs(chain_val - live_val)
            status = '✅' if d < 0.5 else '⚠️'
            print(f'    {status} {short:<43} {chain_val:10.3f} {live_val:10.3f} {d:8.4f}')

    crosscheck_passed = diff < 5.0

except ImportError as e:
    print(f'\n  [SKIP] Required packages not available: {e}')
    crosscheck_passed = None
except Exception as e:
    print(f'\n  [ERROR] Cross-check failed: {e}')
    import traceback
    traceback.print_exc()
    crosscheck_passed = False


---


In [ ]:
# ============================================================
# §8. EXPORT — NB05_A1_results.json
# ============================================================
import json
import os as _os

# ── Read Cobaya checkpoint for official R-1 ──
cobaya_r1_raw = None
for _cp_try in [f'/content/{CHAIN_ROOT}.checkpoint', f'{CHAIN_ROOT}.checkpoint']:
    if _os.path.exists(_cp_try):
        with open(_cp_try) as _cpf:
            for _line in _cpf:
                if 'Rminus1_last' in _line:
                    cobaya_r1_raw = float(_line.split(':')[1].strip())
                    print(f'  [CHECKPOINT] R-1 cobaya = {cobaya_r1_raw:.6f}')
        break
if cobaya_r1_raw is None:
    print('  [WARN] No .checkpoint file — using post-burnin R-1 as cobaya value')
    cobaya_r1_raw = r1_max

print('=' * 60)
print('EXPORT')
print('=' * 60)

def get_mean(name):
    return float(r1_results[name]['mean'])

def get_std(name):
    return float(r1_results[name]['std'])

def get_bf(name):
    if name in header and header.index(name) < len(bestfit):
        return float(bestfit[header.index(name)])
    elif name in r1_results:
        return r1_results[name]['mean']  # post-hoc: bestfit = mean
    return float('nan')

def get_r1_val(name):
    r = r1_results.get(name, {}).get('R-1', np.nan)
    return None if np.isnan(r) else round(float(r), 6)

# Confidence intervals via weighted percentiles
def weighted_percentile(data, weights, percentiles):
    sorted_idx = np.argsort(data)
    sorted_data = data[sorted_idx]
    sorted_weights = weights[sorted_idx]
    cumsum = np.cumsum(sorted_weights)
    cumsum /= cumsum[-1]
    return np.interp(percentiles, cumsum, sorted_data)

def get_ci(name, levels=[0.025, 0.16, 0.50, 0.84, 0.975]):
    if name not in header:
        # Post-hoc constant (e.g. fcdm_z0, I_GKI) — no distribution
        m = r1_results[name]['mean']
        return {'median': m, '68_lower': m, '68_upper': m, '95_lower': m, '95_upper': m}
    idx = header.index(name)
    vals = all_data[:, idx]
    pcts = weighted_percentile(vals, all_w, levels)
    return {
        'median': round(float(pcts[2]), 6),
        '68_lower': round(float(pcts[1]), 6),
        '68_upper': round(float(pcts[3]), 6),
        '95_lower': round(float(pcts[0]), 6),
        '95_upper': round(float(pcts[4]), 6),
    }

# Chi2 bestfit dict
chi2_bf = {}
for k in header:
    if k.startswith('chi2__') and k != 'chi2':
        chi2_bf[k.replace('chi2__', '')] = get_bf(k)
chi2_bf['total'] = float(total_from_col)

# N_DATA from config likelihoods (verified against Cobaya log)
# CamSpec prints "Number of data points: 9915" in the log.
# lowl, lensing, BAO, SNe from official Cobaya docs.
N_DATA_PER_LIK = {}
for lik_name in config.get('likelihood', {}).keys():
    if 'CamSpec' in lik_name and 'TTTEEE' in lik_name:
        N_DATA_PER_LIK[lik_name] = 9915   # from Cobaya log
    elif lik_name == 'planck_2018_lowl.TT':
        N_DATA_PER_LIK[lik_name] = 28     # ell 2-29
    elif lik_name == 'planck_2018_lowl.EE':
        N_DATA_PER_LIK[lik_name] = 396    # SimAll bins
    elif lik_name == 'planck_2018_lensing.clik':
        N_DATA_PER_LIK[lik_name] = 9      # phi bandpowers
    elif lik_name == 'bao.desi_dr2':
        N_DATA_PER_LIK[lik_name] = 12     # DESI DR2 data points
    elif lik_name == 'sn.pantheonplus':
        N_DATA_PER_LIK[lik_name] = 1701   # PantheonPlus SNe
    # eu_derived.EU_Derived: logp=0, not a constraint

n_data_total = sum(N_DATA_PER_LIK.values())
n_sampled = len([p for p, v in config['params'].items()
                 if isinstance(v, dict) and 'prior' in v])
n_dof = n_data_total - n_sampled

# Post-hoc derived params if not in chains
# Void boost computed from formula (not hardcoded)
# Wu & Huterer 2017 Eq. 7 + Marra+ 2013 Θ correction (PRL 110, 241305)
H0_GKI_BASE = 68.90             # NB02 analytic H0_GKI (2026-06-15 audit)
_delta_obs_KBC = -0.46           # Keenan, Barger & Cowie (2013)
_f_growth_ref = 0.5140           # Riccati ODE at Omega_m_EU (NB02 §4.1, exact)
_delta_true_ref = _delta_obs_KBC / (1.0 + _f_growth_ref)  # RSD correction (Haslbauer+ 2020)
_Theta_ref = 1.0 - 0.0882 * _delta_true_ref - 0.123 * np.sin(_delta_true_ref) / (1.29 + _delta_true_ref)
dH0_VOID_BASE = -(1.0/3.0) * _delta_true_ref * _f_growth_ref * _Theta_ref * H0_GKI_BASE

if 'H0' in r1_results:  # FORCE recompute H0_LKI with Theta (overrides chain value)
    h0_col = header.index('H0')
    h0_vals = all_data[:, h0_col]
    h0_lki_vals = h0_vals + dH0_VOID_BASE * (h0_vals / H0_GKI_BASE)
    h0_lki_mean = np.average(h0_lki_vals, weights=all_w)
    h0_lki_std = np.sqrt(np.average((h0_lki_vals - h0_lki_mean)**2, weights=all_w))
    r1_results['H0_LKI'] = {'R-1': r1_results['H0']['R-1'], 'mean': h0_lki_mean, 'std': h0_lki_std}
    if 'H0_LKI' in header:
        _lki_idx = header.index('H0_LKI')
        all_data[:, _lki_idx] = h0_lki_vals
    else:
        all_data = np.column_stack([all_data, h0_lki_vals])
        header.append('H0_LKI')
    bf_h0 = float(bestfit[header.index('H0') if 'H0' in header[:len(bestfit)] else 0])
    bf_h0_lki = bf_h0 + dH0_VOID_BASE * (bf_h0 / H0_GKI_BASE)
    if 'H0_LKI' in header and header.index('H0_LKI') < len(bestfit):
        bestfit[header.index('H0_LKI')] = bf_h0_lki  # overwrite
    else:
        bestfit = np.append(bestfit, bf_h0_lki)
    print(f'  [COMPUTED] H0_LKI = {h0_lki_mean:.3f} ± {h0_lki_std:.3f} (post-hoc from H0)')

# Helper to extract float from eu_param_vals (may be dict with 'value' key)
def _eu_float(key, default):
    v = eu_param_vals.get(key, default)
    if isinstance(v, dict):
        return float(v.get('value', v.get('ref', v.get('mean', default))))
    return float(v) if v is not None else default

if 'fcdm_z0' not in r1_results:  # Only compute if not in chains (free-param runs keep posteriors)
    # Correct formula: fcdm = exp(-λ · I_GKI) via numerical integration
    eps = _eu_float('eu_epsilon_ir', 0.04264)
    zt = _eu_float('eu_z_trans', 5.986)
    bb = _eu_float('eu_b', 0.52778)
    lam = 2.0/3.0
    _z_grid = np.linspace(0, 50, 2000)
    _eps_z = eps / (1.0 + ((1.0 + _z_grid) / (1.0 + zt))**(1.0 / bb))
    _eps_z[_z_grid > zt] = 0.0
    _trapz_fn = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
    _I_GKI_val = _trapz_fn(_eps_z / (1.0 + _z_grid), _z_grid)
    fcdm = np.exp(-lam * _I_GKI_val)
    r1_results['fcdm_z0'] = {'R-1': float('nan'), 'mean': fcdm, 'std': 0.0}
    print(f'  [COMPUTED] fcdm_z0 = {fcdm:.6f} (exp(-λ·I_GKI), numerical)')

if 'I_GKI' not in r1_results:  # Only compute if not in chains (free-param runs keep posteriors)
    eps = _eu_float('eu_epsilon_ir', 0.04264)
    zt = _eu_float('eu_z_trans', 5.986)
    bb = _eu_float('eu_b', 0.52778)
    lam = 2.0/3.0
    _z_grid2 = np.linspace(0, 50, 2000)
    _eps_z2 = eps / (1.0 + ((1.0 + _z_grid2) / (1.0 + zt))**(1.0 / bb))
    _eps_z2[_z_grid2 > zt] = 0.0
    _trapz_fn2 = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
    I_val = _trapz_fn2(_eps_z2 / (1.0 + _z_grid2), _z_grid2)
    r1_results['I_GKI'] = {'R-1': float('nan'), 'mean': I_val, 'std': 0.0}
    print(f'  [COMPUTED] I_GKI = {I_val:.6f} (numerical ∫ε/(1+z)dz)')

# Tensions
H0_SHOES, H0_SHOES_ERR = 73.17, 0.86
S8_DES, S8_DES_ERR = 0.776, 0.017

# Build param entries with full info
def param_entry(name):
    entry = {
        'mean': round(get_mean(name), 6),
        'std': round(get_std(name), 6),
        'bestfit': round(get_bf(name), 6),
        'R-1': get_r1_val(name),
    }
    entry.update(get_ci(name))
    return entry

# ESS from GetDist
ess_dict = {}
for p in header[2:]:
    idx = header[2:].index(p)
    try:
        ess = combined_gd.getEffectiveSamplesForParamIndex(idx)
        ess_dict[p] = int(ess)
    except:
        pass

# Perturbation flag from YAML
eu_pert_flag = extra_args.get('eu_has_ide_perturbations', 'NOT_SET')

# Full results
results = {
    '_metadata': {
        'notebook': 'NB05_MCMC_A1',
        'run_label': f'A1 ({CHAIN_ROOT}, eu_has_ide_perturbations={eu_pert_flag})',
        'run_description': eu_pert_label,
        'date': str(np.datetime64('now')),
        'source': f'{CHAIN_ROOT} ({N_CHAINS} chains)',
        'convergence': {
            'R-1_max_post_burnin': round(r1_max, 6),
            'R-1_max_cobaya': round(float(cobaya_r1_raw), 6) if cobaya_r1_raw is not None else None,
            'target': float(sampler_cfg.get('Rminus1_stop', 0.01)),
            'status': ('FULLY_CONVERGED' if cobaya_r1_raw is not None and cobaya_r1_raw < 0.01 else 'CONVERGED' if cobaya_r1_raw is not None and cobaya_r1_raw < 0.03 else 'NOT_CONVERGED'),
        },
        'samples_weighted': total_weighted,
        'effective_sample_size': ess_dict,
        'n_chains': N_CHAINS,
        'burn_in': '30%',
        'crosscheck': 'PASSED' if crosscheck_passed else 'FAILED',
        'software': {
            'cobaya': cobaya.__version__,
            'getdist': getdist.__version__,
            'class': config.get('theory', {}).get('classy', {}).get('version', 'v3.3.4'),
            'eu_patch': 'vf1.3',
        },
        'likelihoods': likelihoods,
        'n_data_per_likelihood': N_DATA_PER_LIK,
        'n_data_total': n_data_total,
        'n_sampled_params': n_sampled,
        'n_dof': n_dof,
        'chi2_per_dof': round(float(total_from_col) / n_dof, 4),
    },
    'cosmological_params': {
        p: param_entry(p)
        for p in ['omega_b', 'omega_cdm', 'theta_s_100', 'tau_reio', 'logA', 'n_s', 'A_planck']
        if p in r1_results
    },
    'derived_params': {},
    'convergence_per_param': {
        name: get_r1_val(name)
        for name in r1_results
        if get_r1_val(name) is not None
    },
    'eu_params_fixed': {
        'eps_IR': eu_param_vals.get('eu_epsilon_ir'),
        'z_trans': eu_param_vals.get('eu_z_trans'),
        'b': eu_param_vals.get('eu_b'),
        'lambda': extra_args.get('eu_lambda', 0.66667),
        'eu_has_ide_perturbations': eu_pert_flag,
    },
    'chi2_bestfit': chi2_bf,
    'chi2_per_dof': round(float(total_from_col) / n_dof, 4),
    'tensions': {
        'H0_LKI_vs_SH0ES_sigma': round(abs(get_mean('H0_LKI') - H0_SHOES) / H0_SHOES_ERR, 2),
        'H0_GKI_vs_SH0ES_sigma': round(abs(get_mean('H0') - H0_SHOES) / H0_SHOES_ERR, 2),
        'S8_vs_DES_sigma': round(abs(get_mean('S8') - S8_DES) / S8_DES_ERR, 2),
    },
}

# Derived params with full stats
for p in ['H0', 'sigma8', 'S8', 'Omega_m', 'fcdm_z0', 'H0_LKI', 'I_GKI', 'rdrag']:
    if p in r1_results:
        entry = param_entry(p)
        if p == 'rdrag':
            entry['unit'] = 'Mpc'
        results['derived_params'][p] = entry

with open('NB05_A1_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)
print(f'  [SAVED] NB05_A1_results.json')

# Summary
print(f'\n  === Key Results ===')
print(f'  IDE perturbations: eu_has_ide_perturbations = {eu_pert_flag}')
print(f'    \u2192 {eu_pert_label}')
print(f'  H0     = {get_mean("H0"):.3f} \u00b1 {get_std("H0"):.3f}  [{get_ci("H0")["95_lower"]:.2f}, {get_ci("H0")["95_upper"]:.2f}] (95% CL)')
print(f'  H0_LKI = {get_mean("H0_LKI"):.3f} \u00b1 {get_std("H0_LKI"):.3f}  [{get_ci("H0_LKI")["95_lower"]:.2f}, {get_ci("H0_LKI")["95_upper"]:.2f}] (95% CL)')
print(f'  S8     = {get_mean("S8"):.4f} \u00b1 {get_std("S8"):.4f}  [{get_ci("S8")["95_lower"]:.4f}, {get_ci("S8")["95_upper"]:.4f}] (95% CL)')
print(f'  \u03c7\u00b2/dof = {total_from_col:.1f} / {n_dof} = {total_from_col/n_dof:.4f}')
print(f'\n  === Tensions ===')
print(f'  H0_LKI vs SH0ES: {results["tensions"]["H0_LKI_vs_SH0ES_sigma"]:.2f}\u03c3')
print(f'  H0_GKI vs SH0ES: {results["tensions"]["H0_GKI_vs_SH0ES_sigma"]:.2f}\u03c3')
print(f'  S8 vs DES:       {results["tensions"]["S8_vs_DES_sigma"]:.2f}\u03c3')


---


In [ ]:
# ============================================================
# §9. DOWNLOAD
# ============================================================
import zipfile, glob

_base = '/content'
zip_path = os.path.join(_base, 'NB05_A1_outputs.zip')
_count = 0

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for jpath in [os.path.join(_base, 'NB05_A1_results.json'), 'NB05_A1_results.json']:
        if os.path.exists(jpath):
            zf.write(jpath, 'NB05_A1_results.json')
            print(f'  Added: NB05_A1_results.json')
            _count += 1
            break
    for _fdir in [os.path.join(_base, 'figures'), 'figures']:
        if os.path.isdir(_fdir):
            for f in sorted(glob.glob(os.path.join(_fdir, 'fig_NB05_A1_*'))):
                arcname = os.path.join('figures', os.path.basename(f))
                zf.write(f, arcname)
                print(f'  Added: {arcname}')
                _count += 1
            break

assert _count > 0, 'FATAL: No files added to zip!'
print(f'\n[OK] NB05_A1_outputs.zip - {_count} files ({os.path.getsize(zip_path):,} bytes)')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f'Local env - file at {zip_path}')


---

## References

1. **Planck NPIPE** — CamSpec TTTEEE + lowl TT/EE + lensing
2. **DESI DR2** — BAO measurements (2024)
3. **Pantheon+** — Type Ia supernovae (Brout+ 2022)
4. **Cobaya** — Torrado & Lewis (2021)
5. **GetDist** — Lewis (2019)
6. **CLASS** — Blas, Lesgourgues & Tram (2011)
7. **Alvim (2025, 2026)** — Evaporating Universe, Paper I
